# specfp8 — Compatibility Probe
**Speculative Decoding × FP8 Compatibility Study**

This notebook runs Phase 1: the compatibility matrix on an L4 GPU (SM89).

**Before running:** Go to Runtime → Change runtime type → Select **L4 GPU**

In [ ]:
# Cell 1: Verify GPU
import subprocess, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    name = torch.cuda.get_device_name()
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / 1024**3
    print(f'\nGPU: {name}')
    print(f'Compute capability: {cap[0]}.{cap[1]}')
    print(f'VRAM: {vram:.1f} GB')
    if cap >= (8, 9):
        print('\n✓ GPU is Ada-class (SM89+). Full FP8 probe will run.')
    else:
        print(f'\n⚠ GPU is SM{cap[0]}{cap[1]}, not SM89+.')
        print('  FP8 cells will be skipped. BF16 smoke test will still work.')
        print('  For the full probe, use an L4/L40S/RTX 4090.')
else:
    print('✗ No CUDA GPU. Go to Runtime → Change runtime type → L4 GPU')

In [ ]:
# Cell 2: Clone repo and install harness
!git clone https://github.com/akshathtiwari/spec-fp8-study.git
%cd spec-fp8-study
!pip install -e . -q
print('\n✓ Harness installed')

In [ ]:
# Cell 3: Install vLLM (~3-5 min)
!pip install vllm -q
import vllm
print(f'\n✓ vLLM {vllm.__version__} installed')

In [ ]:
# Cell 4: Smoke test — 2 cells, ~10 min
# Tests none + ngram on BF16 with Qwen3-0.6B (tiny model)
!python -m specfp8.probe --sweep sweeps/test_mini.yaml --results results/

# Show results
import json
print('\n--- Results ---')
with open('results/cells.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        print(f"{rec['cell_id']}: {rec['outcome']['status']} "
              f"(mechanism={rec['config']['mechanism']})")

In [ ]:
# Cell 5: Create vLLM-only sweep (full matrix, vLLM engine only)
import yaml

with open('sweeps/compat.yaml') as f:
    spec = yaml.safe_load(f)

spec['axes']['engine'] = ['vllm']

with open('sweeps/compat_vllm.yaml', 'w') as f:
    yaml.dump(spec, f, default_flow_style=False)

# Count cells
from specfp8.cells import expand
cells = expand('sweeps/compat_vllm.yaml')
print(f'vLLM-only sweep: {len(cells)} cells')
for c in cells:
    print(f'  {c.mechanism:<8} weight={c.weight_precision} kv={c.kv_cache_dtype}')

In [ ]:
# Cell 6: Run full vLLM probe (~20 cells, ~2 hrs)
# Resume-safe: re-running skips completed cells
!python -m specfp8.probe --sweep sweeps/compat_vllm.yaml --results results/

In [ ]:
# Cell 7: Check results
import json

statuses = {}
with open('results/cells.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        cfg = rec['config']
        key = f"{cfg['mechanism']:<8} | {cfg['weight_precision']:<4} | {cfg['kv_cache_dtype']:<10}"
        status = rec['outcome']['status']
        tau = rec.get('spec', {}).get('tau', '-')
        statuses[key] = (status, tau)

print(f"{'Config':<35} {'Status':<15} {'τ'}")
print('-' * 60)
for k, (s, t) in sorted(statuses.items()):
    marker = '✓' if s == 'ok' else '✗'
    tau_str = f'{t:.2f}' if isinstance(t, (int, float)) else str(t)
    print(f'{k:<35} {marker} {s:<13} {tau_str}')

# H1 check
print('\n--- H1 Check ---')
for line in open('results/cells.jsonl'):
    rec = json.loads(line)
    cfg = rec['config']
    if cfg['mechanism'] == 'dflash' and cfg['kv_cache_dtype'] == 'fp8_e4m3':
        status = rec['outcome']['status']
        error = rec['outcome'].get('error_verbatim', '')[:200] if rec['outcome'].get('error_verbatim') else 'None'
        print(f"DFlash + FP8-KV: {status}")
        print(f"Error: {error}")
        if status != 'ok':
            print('\n→ H1 CONFIRMED: headline A')
        else:
            print('\n→ H1 REFUTED: headline B (measurement study)')

In [ ]:
# Cell 8: Backup results to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
backup_dir = '/content/drive/MyDrive/specfp8-results'
os.makedirs(backup_dir, exist_ok=True)
shutil.copytree('results', backup_dir, dirs_exist_ok=True)
print(f'\n✓ Results backed up to Google Drive: {backup_dir}')

In [ ]:
# Cell 9: Download results locally
from google.colab import files
!tar czf probe_results.tar.gz results/
files.download('probe_results.tar.gz')

---
## SGLang cells (run in a SEPARATE session)

Start a **new Colab runtime** (to avoid vLLM/SGLang conflicts),
then run the cells below.

The resume logic will skip all completed vLLM cells automatically.

In [ ]:
# SGLang Cell 1: Setup
# Uncomment and run in a fresh session

# !git clone https://github.com/akshathtiwari/spec-fp8-study.git
# %cd spec-fp8-study
# !pip install -e . -q
# !pip install "sglang[all]" -q

# # Restore previous results from Drive
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree('/content/drive/MyDrive/specfp8-results', 'results', dirs_exist_ok=True)
# print('Restored previous results')

In [ ]:
# SGLang Cell 2: Run SGLang probe
# Uncomment and run after SGLang Cell 1

# import yaml
# with open('sweeps/compat.yaml') as f:
#     spec = yaml.safe_load(f)
# spec['axes']['engine'] = ['sglang']
# with open('sweeps/compat_sglang.yaml', 'w') as f:
#     yaml.dump(spec, f)
# !python -m specfp8.probe --sweep sweeps/compat_sglang.yaml --results results/